# VRDet: Real-Time Intraday Volatility Regime Classifier
## Stage 01: Load Data

Raw data is downloaded **locally** by `scripts/fetch_data.py` (Angel One SmartAPI + yfinance) and uploaded to
Google Drive at `MyDrive/vrdet/data/raw/`. This notebook loads those parquet files.

In [ ]:
%pip install -q pyarrow gdown hmmlearn arch lightgbm xgboost

### Locate data
`drive.mount()` does not work from the VS Code Colab extension (its auth popup can't complete), so data is pulled
from the Drive folder through a **share link** with `gdown` instead. In Drive: right-click `vrdet/data/raw` →
Share → General access: *Anyone with the link* (Viewer) → copy link into `DRIVE_FOLDER_URL`.
If you run this notebook in the Colab web UI, set `USE_DRIVE_MOUNT = True` instead.

In [ ]:
from pathlib import Path

DRIVE_FOLDER_URL = ''  # share link of MyDrive/vrdet/data/raw
USE_DRIVE_MOUNT = False

if USE_DRIVE_MOUNT:
    from google.colab import drive
    drive.mount('/content/drive')
    RAW_DIR = Path('/content/drive/MyDrive/vrdet/data/raw')
else:
    import gdown
    RAW_DIR = Path('/content/vrdet_data/raw')
    RAW_DIR.mkdir(parents=True, exist_ok=True)
    gdown.download_folder(DRIVE_FOLDER_URL, output=str(RAW_DIR), quiet=True)

FILES = sorted(RAW_DIR.rglob('*.parquet'))  # rglob: gdown may nest files in a subfolder
assert FILES, f'No parquet files under {RAW_DIR}. Check DRIVE_FOLDER_URL / sharing settings.'
print('Data dir:', RAW_DIR)
print([str(p.relative_to(RAW_DIR)) for p in FILES])

### Load and check

In [ ]:
import pandas as pd

data = {p.stem: pd.read_parquet(p) for p in FILES}

BARS_PER_DAY = {'1m': 375, '5m': 75, '15m': 25, '1h': 7}
for name, df in data.items():
    if df.empty:
        print(f'{name}: NO DATA'); continue
    days = df.groupby(df.index.date).size()
    line = f'{name:14s} {len(df):>9,} rows  {df.index[0]} -> {df.index[-1]}  {len(days):,} sessions'
    expected = BARS_PER_DAY.get(name.rsplit('_', 1)[-1])
    if expected:
        line += f'  short sessions: {(days < expected).sum()}'
    print(line)

### First look
NIFTY close, 15-minute realized volatility (rolling 1 day) and India VIX.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

nifty, vix = data['NIFTY_15m'], data['INDIAVIX_15m']
r = np.log(nifty['close']).diff()
rv = r.rolling(25).std() * np.sqrt(25 * 252) * 100  # annualised %

fig, ax = plt.subplots(3, 1, figsize=(14, 9), sharex=True)
ax[0].plot(nifty.index, nifty['close'], lw=0.6); ax[0].set_ylabel('NIFTY')
ax[1].plot(rv.index, rv, lw=0.6, color='tab:red'); ax[1].set_ylabel('RV 1d (ann. %)')
ax[2].plot(vix.index, vix['close'], lw=0.6, color='tab:purple'); ax[2].set_ylabel('India VIX')
plt.tight_layout(); plt.show()

# Intraday volatility seasonality: mean |return| by time of day
r.abs().groupby(r.index.time).mean().plot(kind='bar', figsize=(14, 3), title='Mean |15m return| by bar time');

### Data quality: India VIX intraday
India VIX has never traded below ~9, so values below that are errors. Three checks:
1. Summary statistics: the minimum and the largest bar-to-bar jumps.
2. Compare intraday bars with the daily VIX series on the same date.
3. Count *flat* sessions (at most 2 distinct closes all day), which a live volatility index never produces.

In [ ]:
vix15, vix1d = data['INDIAVIX_15m'], data['INDIAVIX_1d']
c = vix15['close']
print('1) min:', c.min(), '| max:', c.max(), '| largest |log change| in one bar:', round(np.log(c).diff().abs().max(), 2))

day = '2015-12-01'
print()
print(f'2) {day}: intraday closes', sorted(c.loc[day].unique()), 'vs daily close', vix1d['close'].loc[day:day].iloc[0])

g = vix15.groupby(vix15.index.date)['close'].agg(['min', 'max', 'nunique'])
bad = g[(g['min'] < 2) | (g['nunique'] <= 2)]
print()
print(f'3) invalid sessions: {len(bad)}, from {bad.index.min()} to {bad.index.max()}')
print('   invalid sessions after 2016-09-21:', [str(d) for d in bad.index if str(d) > '2016-09-21'])

fig, ax = plt.subplots(figsize=(14, 3))
ax.plot(c.index, c, lw=0.5, label='Angel One 15m close')
ax.plot(pd.to_datetime(vix1d.index), vix1d['close'], lw=0.8, alpha=0.7, label='daily close')
ax.set_yscale('log'); ax.set_title('India VIX: intraday feed vs daily (log scale)'); ax.legend(); plt.show()

**Conclusion:** Angel One's intraday India VIX from 2015-12-01 to 2016-09-21 is corrupt (the daily value scaled by 1/100 and flat all session, then integer-rounded and flat), and isolated later sessions (e.g. 2021-03-01) are broken. From 2017 onward, 99-100% of sessions reproduce the official daily values, and NIFTY candles are 99.6-100% valid in every year.

The general rule, applied to every session of every series (`vrdet.quality.session_report`, also run at the end of `scripts/fetch_data.py`): a session is valid if it is not flat, has at least 13 bars, and its candles reproduce at least 2 of the official daily high / low / close within 1%. Invalid NIFTY sessions are dropped; invalid VIX sessions are treated as missing.
